# Part 1: agentic retrieval で knowledge base を構築する

Zava のアクセスバッジとノート PC を受け取ったばかりのあなた。最初の仕事は、社内の HR や福利厚生に関する質問に答えるための AI 駆動の knowledge system を構築することです。

**🎯 ミッション**
- クラウドアーキテクチャの PDF を **File Knowledge Source** に直接アップロードしてクエリする
- ビルド済みの 2 つの検索インデックス（**HR docs** と **health benefits docs**）を **Indexed Knowledge Sources** として接続する
- 各サブクエスチョンを自動的に適切なソースへルーティングする、マルチソース **Knowledge Base** を構築する


## 構成要素

Foundry IQ（Azure AI Search）における **knowledge base** とは、chat completion model と 1 つ以上の knowledge source を結びつけるトップレベルのリソースです。どのデータソースをクエリするか、どのモデルで推論するか、そしてクエリ実行をどのように最適化するかを定義します。

このノートブックでは、次の 2 種類の knowledge source を扱います。

1. **File Knowledge Source**: 生のファイルを直接アップロードします。チャンク化、埋め込み生成、インデックス作成はサービスが自動で行います。
2. **Indexed Knowledge Sources**: ビルド済みの検索インデックスに接続します。スキーマや処理を完全に制御したい本番ワークロード向けです。

## 事前構成済みの環境

このラボでは、以下の Azure リソースが事前にセットアップされています。

* Azure AI Search
  * `hrdocs` インデックス: HR ポリシー、ハンドブック、会社情報
  * `healthdocs` インデックス: 健康関連の福利厚生、保険、補償範囲
  * Semantic ranking 有効、埋め込みは事前計算済み
* Microsoft Foundry Models 経由の Azure OpenAI
  * `gpt-5.4`: 推論と回答生成のための chat completion
  * `text-embedding-3-large`: ベクトル検索用の embedding model
* Microsoft Fabric

## Step 1: 環境変数をセットアップする

Azure リソースの設定を読み込みます。プロジェクトフォルダーの `.env` ファイルには、Azure AI Search のエンドポイント、Azure OpenAI のエンドポイント、モデル構成など必要なものがすべて入っています。認証には **Microsoft Entra ID**（`DefaultAzureCredential`）を使用します。

**ℹ️ 注意**
- 下のセルを初めて実行するとき、kernel の選択を求められます。**Python Environments** を選び、**.venv** 環境を選択してください。
- 「Do you want to allow public and private networks to access this app?」というプロンプトが表示されることもあります。**Allow** を選択してください。
- Entra ID 認証を使用するため、Azure AI Search と Azure OpenAI の両方に対して適切な RBAC ロール（例: **Search Index Data Contributor**、**Cognitive Services OpenAI User**）が割り当てられている必要があります。

**⚠️ トラブルシューティング:**
コードセルがハングしてスピナーが止まらない場合は、次の手順を試してください。
1. ノートブック上部のツールバーから **Restart** を選択する。それでも解決しない場合:
2. VS Code のコマンドパレットから **Reload window** を実行する。それでも解決しない場合:
3. VS Code を完全に閉じてから起動し直す。

In [16]:
import json
import os

from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv

load_dotenv(override=True)

AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-05-01-preview"
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_OPENAI_EMBEDDING_DEPLOYMENT = os.environ.get("AZURE_OPENAI_EMBEDDING_DEPLOYMENT", "text-embedding-3-large")
AZURE_OPENAI_EMBEDDING_MODEL = "text-embedding-3-large"

HRDOCS_INDEX = "hrdocs"
HEALTHDOCS_INDEX = "healthdocs"

# --- SharePoint 接続設定（.env から読み込み）---
SHAREPOINT_SITE_URL = os.environ.get(
    "SHAREPOINT_SITE_URL",
    "https://mngenvmcap097540.sharepoint.com/sites/it-ops-knowledge-demo",
)
SHAREPOINT_APPLICATION_ID = os.environ["SHAREPOINT_APPLICATION_ID"]
SHAREPOINT_TENANT_ID = os.environ["SHAREPOINT_TENANT_ID"]
# シークレットレス（マネージド ID + フェデレーション ID 資格情報）認証用。
# ★ 検索サービスのシステム割り当てマネージド ID 自身のアプリ ID を指定する点に注意。
#    （取り込み用アプリ ID = SHAREPOINT_APPLICATION_ID とは別物）
SHAREPOINT_FEDERATED_CREDENTIAL_APP_ID = os.environ["SHAREPOINT_FEDERATED_CREDENTIAL_APP_ID"]

credential = DefaultAzureCredential()
print("Environment variables loaded (using Entra ID authentication)")


Environment variables loaded (using Entra ID authentication)


---

## A: File Knowledge Source

**File Knowledge Source** を使うと、生のファイルを直接アップロードできます。チャンク化、埋め込み生成、インデックス作成はサービスが自動で行います。knowledge base にドキュメントを入れる最も手早い方法です。

## Step 2: File Knowledge Source を作成する

embedding model を指定した file knowledge source を構成し、自動取り込みを有効化します。`ingestion_parameters` では次を指定します。
- **Embedding model**: アップロードされたコンテンツのベクトル埋め込みを生成します
- **Content extraction mode**: ドキュメントからテキストを抽出する方法

In [2]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    FileKnowledgeSource,
    FileKnowledgeSourceParameters,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeSourceAzureOpenAIVectorizer,
    KnowledgeSourceIngestionParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "sharepoint-docs-knowledge-source"

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=credential)

file_ks = FileKnowledgeSource(
    name=FILE_KNOWLEDGE_SOURCE_NAME,
    description="IT運用保守ナレッジポータルのSharePointコンテンツ用ナレッジソース",
    file_parameters=FileKnowledgeSourceParameters(
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                )
            ),
            content_extraction_mode="minimal",
            )
    ),
)

index_client.create_or_update_knowledge_source(file_ks)
print(f"File knowledge source '{FILE_KNOWLEDGE_SOURCE_NAME}' created or updated successfully.")

File knowledge source 'sharepoint-docs-knowledge-source' created or updated successfully.


## Step 3-a: knowledge source にファイルをアップロードする

ドキュメントを file knowledge source に直接アップロードします。サービスがテキストを自動抽出し、検索可能なセグメントに分割し、ベクトル埋め込みを生成します。

⏱️ サンプルファイルの取り込みには約 40 秒かかります。

In [ ]:
from pathlib import Path

data_dir = Path("..") / "data" / "ai-search-data"
files_to_upload = [
    data_dir / "filesource" / "MSFT_cloud_architecture_zava.pdf"
]

uploaded_files = []
for file_path in files_to_upload:
    with open(file_path, "rb") as f:
        file_content = f.read()
    uploaded = index_client.upload_knowledge_source_file(
        FILE_KNOWLEDGE_SOURCE_NAME, file_content, filename=file_path.name
    )
    uploaded_files.append(uploaded)
    print(f"Uploaded: {file_path.name} (file_id: {uploaded.file_id}, {uploaded.file_size_bytes} bytes)")

print(f"\nTotal files uploaded: {len(uploaded_files)}")

## Step 3-b: SharePoint サイトをマネージド ID 経由で接続する

ローカルファイルをアップロードする代わりに、社内の **IT運用保守ナレッジポータル**（SharePoint サイト）に直接接続し、そのドキュメントライブラリを Azure AI Search に取り込みます。ここでは **Indexed SharePoint Knowledge Source** を作成します。これは内部でデータソース・スキルセット・インデクサー・インデックスを自動生成し、SharePoint のコンテンツをチャンク化・ベクトル化して検索インデックスに格納する **インデックス付き** のナレッジソースです。

認証には **シークレットレス（フェデレーション ID 資格情報 / マネージド ID）** を使用します。クライアントシークレットを持たず、Azure AI Search の **システム割り当てマネージド ID** が Entra アプリのフェデレーション ID 資格情報（FIC）を通じて SharePoint へのアクセストークンを取得します。

**🔑 事前準備（管理者作業）**
この方式を使うには、次が構成済みである必要があります（[SharePoint インデクサーの前提条件](https://learn.microsoft.com/azure/search/search-how-to-index-sharepoint-online#prerequisites) を参照）。
1. Azure AI Search で **システム割り当てマネージド ID** を有効化する
2. SharePoint ドキュメント読み取り権限を持つ **Microsoft Entra アプリ登録** を作成する（アプリケーション権限: `Sites.Read.All` など）
3. そのアプリ登録に、Search サービスのマネージド ID を信頼する **フェデレーション ID 資格情報（FIC）** を構成する

**ℹ️ 実装に必要な値**
下のセルは `.env` から次の値を読み込みます。お手元の環境に合わせて設定してください。
- `SHAREPOINT_SITE_URL`: 接続先サイトの URL（例: `https://mngenvmcap097540.sharepoint.com/sites/it-ops-knowledge-demo`）
- `SHAREPOINT_APPLICATION_ID`: 取り込み用 Entra アプリの **アプリケーション (クライアント) ID**（GUID）
- `SHAREPOINT_TENANT_ID`: SharePoint サイトの **テナント ID**（GUID）
- `SHAREPOINT_FEDERATED_CREDENTIAL_APP_ID`: シークレットレス認証で FIC がフェデレートする Entra アプリのクライアント ID（通常は `SHAREPOINT_APPLICATION_ID` と同じ。未設定の場合は自動的に同じ値を使用します）

> 💡 クライアントシークレット方式を使う場合は、接続文字列の `FederatedCredentialApplicationId=...` を `ApplicationSecret=[シークレット]` に置き換えます。

In [17]:
from azure.search.documents.indexes.models import (
    IndexedSharePointContainerName,
    IndexedSharePointKnowledgeSource,
    IndexedSharePointKnowledgeSourceParameters,
    KnowledgeSourceContentExtractionMode,
)

SHAREPOINT_KNOWLEDGE_SOURCE_NAME = "sharepoint-it-ops-knowledge-source"

# シークレットレス接続文字列（アプリケーション権限 + FIC）。
# - ApplicationId: Graph 権限と FIC を構成した「取り込み用アプリ」のクライアント ID
# - FederatedCredentialApplicationId: 「検索サービスのシステム割り当て
#   マネージド ID 自身のアプリ ID」。これを入れないと委任権限モードと解釈され、
#   バックグラウンドのインデクサーがデバイスコードサインインを要求して失敗します。
#   また、この値が MI のアプリ ID と一致しないと作成時に検証エラーになります。
sharepoint_connection_string = (
    f"SharePointOnlineEndpoint={SHAREPOINT_SITE_URL};"
    f"ApplicationId={SHAREPOINT_APPLICATION_ID};"
    f"FederatedCredentialApplicationId={SHAREPOINT_FEDERATED_CREDENTIAL_APP_ID};"
    f"TenantId={SHAREPOINT_TENANT_ID}"
)

sharepoint_ks = IndexedSharePointKnowledgeSource(
    name=SHAREPOINT_KNOWLEDGE_SOURCE_NAME,
    description="IT運用保守ナレッジポータル（SharePoint）のインデックス付きナレッジソース",
    indexed_share_point_parameters=IndexedSharePointKnowledgeSourceParameters(
        connection_string=sharepoint_connection_string,
        container_name=IndexedSharePointContainerName.DEFAULT_SITE_LIBRARY,
        ingestion_parameters=KnowledgeSourceIngestionParameters(
            embedding_model=KnowledgeSourceAzureOpenAIVectorizer(
                azure_open_ai_parameters=AzureOpenAIVectorizerParameters(
                    resource_url=AZURE_OPENAI_ENDPOINT,
                    deployment_name=AZURE_OPENAI_EMBEDDING_DEPLOYMENT,
                    model_name=AZURE_OPENAI_EMBEDDING_MODEL,
                )
            ),
            content_extraction_mode=KnowledgeSourceContentExtractionMode.MINIMAL,
            disable_image_verbalization=True,
        ),
    ),
)

created = index_client.create_or_update_knowledge_source(sharepoint_ks)
print(f"Indexed SharePoint knowledge source '{SHAREPOINT_KNOWLEDGE_SOURCE_NAME}' created or updated successfully.")
print(f"  kind: {created.kind}")
print(f"  site: {SHAREPOINT_SITE_URL}")


Indexed SharePoint knowledge source 'sharepoint-it-ops-knowledge-source' created or updated successfully.
  kind: indexedSharePoint
  site: https://mngenvmcap097540.sharepoint.com/sites/it-ops-knowledge-demo


### 取り込みステータスを確認する

Indexed SharePoint knowledge source を作成すると、Azure AI Search が **data source・skillset・indexer・index** を自動生成し、インデクサーが SharePoint のコンテンツを **非同期** で取り込みます。下のセルで取り込みの進行状況とエラーを確認できます。

⏱️ サイトのドキュメント量に応じて、初回の取り込み完了までに数分かかることがあります。`status` が `running` の間は時間をおいて再実行してください。

In [18]:
status = index_client.get_knowledge_source_status(SHAREPOINT_KNOWLEDGE_SOURCE_NAME)
print(json.dumps(status.as_dict(), indent=2))

{
  "@odata.context": "https://gi-ai-search.search.windows.net/$metadata#Microsoft.WindowsAzure.Search.Core.Models.V2026_05_01_Preview.KnowledgeSources.KnowledgeSourceStatus",
  "kind": "indexedSharePoint",
  "synchronizationStatus": "active",
  "synchronizationInterval": "1d",
  "currentSynchronizationState": null,
  "lastSynchronizationState": {
    "startTime": "2026-06-14T14:16:46.233Z",
    "endTime": "2026-06-14T14:17:06.437Z",
    "itemsUpdatesProcessed": 45,
    "itemsUpdatesFailed": 0,
    "itemsSkipped": 0,
    "errors": []
  },
  "statistics": {
    "totalSynchronization": 1,
    "averageSynchronizationDuration": "PT20.2041566S",
    "averageItemsProcessedPerSynchronization": 45
  }
}


## Step 4: file source 用の knowledge base を作成する

knowledge base は、次の要素を束ねるオーケストレーション層です。

1. データソース（先ほどの file knowledge source）
2. クエリの理解と回答生成のための LLM（Azure OpenAI）
3. クエリの処理方法とレスポンスのフォーマット設定

この knowledge base は出力モードに `ANSWER_SYNTHESIS` を使用しているため、アタッチした LLM が取得したチャンクから自然な日本語（または元のクエリの言語）で完結した回答を生成します。

In [19]:
from azure.search.documents.indexes.models import (
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalOutputMode,
)

FILE_KNOWLEDGE_BASE_NAME = "sharepoint-it-ops-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
)

knowledge_base = KnowledgeBase(
    name=FILE_KNOWLEDGE_BASE_NAME,
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[
        KnowledgeSourceReference(name=SHAREPOINT_KNOWLEDGE_SOURCE_NAME),
    ],
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{FILE_KNOWLEDGE_BASE_NAME}' created or updated successfully.")

Knowledge base 'sharepoint-it-ops-knowledge-base' created or updated successfully.


## Step 5: file knowledge base にクエリを投げる

アップロードしたドキュメントについて質問してみましょう。knowledge base はファイルの内容を横断検索し、引用付きの自然言語の回答を合成します。

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
)
from IPython.display import Markdown, display

file_kb_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=FILE_KNOWLEDGE_BASE_NAME, credential=credential
)

file_ks_params = FileKnowledgeSourceParams(
    knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = "As a new Zava engineer, what cloud architecture principles should I know about? What data sensitivity levels does Zava use?"

req = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[
                KnowledgeBaseMessageTextContent(text=question)
            ],
        )
    ],
    knowledge_source_params=[file_ks_params],
    include_activity=True,
)

result = file_kb_client.retrieve(retrieval_request=req)
display(Markdown(result.response[0].content[0].text))

### activity log を確認する

activity log には、knowledge base が実行した一連のステップが記録されています。各 activity には次が含まれます。

* `type`: activity の種別。今回のログでは、最初の `modelQueryPlanning`、（file knowledge source へ送られたクエリごとの）複数の `file` ステップ、回答合成ステップの `modelAnswerSynthesis`、semantic re-ranking ステップの `agenticReasoning` が表示されます。
* `elapsedMS`: その activity にかかった時間。
* `id`: activity の ID。各 reference がどの activity から生成されたかをひも付けるのに使えます。reference は次に確認します。

activity のエントリには、種別固有のメタデータも含まれます。これにより、モデル駆動のステップでどれだけトークンが使われたか、取得ステップでどのようなクエリがソースに送られたかを確認できます。

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### references を確認する

次のセルでは、knowledge base が返す生の `references` 配列を表示します。

references を見ると、どのソースから取得したか、各 reference のソースメタデータが分かります。各結果には次が含まれます。

* `type`: この knowledge base は file source のみを含むため、常に `file` です。
* `id`: reference の数値 ID。回答中に `[ref:1]` のような引用マーカーとして登場します。
* `activitySource`: この reference を生成した取得 activity の ID。activity log のどの検索ステップがこのドキュメントを返したかをたどれます。
* `sourceData`: 自動作成された検索インデックスから取得したチャンクのフィールド。file source のインデックスでは常に `uid`、`snippet`、`metadata_storage_path` が含まれます。
* `rerankerScore`: re-ranking モデルによる 0〜4 のスコア。4 が最も関連性が高いことを示します。
* `docName`: アップロードされたファイルに対して自動生成されたファイル名。

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

---

## B: Indexed Knowledge Sources

**Indexed Knowledge Sources** はビルド済みの検索インデックスに接続し、スキーマ、チャンク化、埋め込みを完全に制御できます。大規模データセットを扱う本番ワークロードに最適です。

このセクションでは、取り込み済みのドキュメントチャンクを格納した 2 つの検索インデックスを事前に用意しています。通常はインデクサーパイプラインを構成して、Azure Blob Storage などのソースから検索インデックスへドキュメントを取り込みます。

## Step 6: 検索インデックスを確認する

下のセルを実行して、各インデックスに想定どおりの数のドキュメントチャンクが入っていることを確認してください。

* `hrdocs`: HR ポリシーに関する 50 件のドキュメントチャンク
* `healthdocs`: 健康関連の福利厚生・保険に関する 334 件のドキュメントチャンク

In [ ]:
from azure.search.documents import SearchClient

for index_name in [HRDOCS_INDEX, HEALTHDOCS_INDEX]:
    client = SearchClient(AZURE_SEARCH_SERVICE_ENDPOINT, index_name, credential)
    count = client.get_document_count()
    print(f"{index_name}: {count} documents")


## Step 7: 2 つの indexed knowledge source を作成する

各インデックスを指す knowledge source を 2 つ作成します。

* `healthdocs-knowledge-source`: `healthdocs` インデックスを参照
* `hrdocs-knowledge-source`: `hrdocs` インデックスを参照

いずれも検索対象フィールドとして `snippet` を指定し、引用リンクにパスを使えるよう `blob_path` を source data field として含めます。

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
)

HR_KNOWLEDGE_SOURCE_NAME = "hrdocs-knowledge-source"
HEALTH_KNOWLEDGE_SOURCE_NAME = "healthdocs-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [HR_KNOWLEDGE_SOURCE_NAME, HEALTH_KNOWLEDGE_SOURCE_NAME]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=credential)

for knowledge_source_name, index_name, description in [
    (HR_KNOWLEDGE_SOURCE_NAME, HRDOCS_INDEX, "Zava HR documents"),
    (HEALTH_KNOWLEDGE_SOURCE_NAME, HEALTHDOCS_INDEX, "Zava health benefits documents"),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="snippet"),
            ],
            search_fields=[SearchIndexFieldReference(name="snippet")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)
    print(f"Knowledge source '{knowledge_source.name}' created or updated successfully.")


## Step 8: マルチソースの knowledge base を作成する

両方の search index knowledge source を含む knowledge base を作成します。前回の knowledge base と同様に `output_mode=ANSWER_SYNTHESIS` を使用し、アタッチした LLM で根拠付きの引用を伴う回答を生成します。

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "multisource-search-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Multi-source knowledge base over HR and health document indexes",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
)

index_client.create_or_update_knowledge_base(knowledge_base)
print(f"Knowledge base '{KNOWLEDGE_BASE_NAME}' created or updated successfully.")

## Step 9: マルチソースの knowledge base にクエリを投げる

2 つの knowledge source にまたがる、2 つの観点を含むオンボーディングの質問を投げてみましょう。

* _"How many vacation weeks does a senior Zava employee get?"_ → HR 関連なので `hrdocs` から取得されるはず
* _"Which health plan gives the best coverage for mental health services?"_ → 福利厚生関連なので `healthdocs` から取得されるはず

knowledge base は agentic retrieval を使って次を行います。

1. クエリを分析し、2 つの異なるトピックがあることを認識する
2. 焦点を絞ったサブクエリに分解する
3. 各サブクエリに対して関連する knowledge source を選択する
4. 両方のソースに対して並行に検索を実行する
5. 結果をマージして返す

In [ ]:
from azure.search.documents.knowledgebases.models import (
    SearchIndexKnowledgeSourceParams,
)

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=credential
)

hrdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HR_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)
healthdocs_knowledge_source_params = SearchIndexKnowledgeSourceParams(
    knowledge_source_name=HEALTH_KNOWLEDGE_SOURCE_NAME,
    include_references=True,
    include_reference_source_data=True,
)

question = (
    "I just joined Zava as a senior engineer. "
    "According to company policy, how many vacation weeks does a senior employee get? "
    "And among the available health plans, which gives the best coverage for mental health services?"
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[hrdocs_knowledge_source_params, healthdocs_knowledge_source_params],
    include_activity=True,
)

result = knowledge_base_client.retrieve(retrieval_request=retrieval_request)
display(Markdown(result.response[0].content[0].text))

### activity log を確認する

今回の activity log では、各検索インデックスへ送られたクエリごとに複数の `searchIndex` ステップが表示されます。

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### references を確認する

今回は references の `type` が `searchIndex` になり、`sourceData` のフィールド構成も少し異なります。これらの検索インデックスは、自動作成された file source のインデックスとは異なるフィールド構成になっているためです。

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

上の reference と activity log を見て、次を確認してみてください。

1. **vacation（休暇）** の質問はどの knowledge source が答えましたか？
2. **mental health plan（メンタルヘルスのプラン）** の質問はどの knowledge source が答えましたか？
3. agent は **両方の** ソースをクエリしましたか、それとも各サブクエリに対して最も関連性の高い 1 つだけをクエリしましたか？

## ボーナス: Copilot CLI からクエリする

すべての knowledge base は MCP サーバーのエンドポイントを公開しており、VS Code や GitHub Copilot CLI など MCP 互換のクライアントから追加できます。
Copilot CLI から knowledge base にクエリを投げてみたい場合は、[Copilot CLI sidequest](copilot-cli-sidequest.md) のセットアップ手順に従い、下のコマンドを実行して Zava の knowledge base を MCP サーバーとして利用してください。

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
token = credential.get_token("https://search.azure.com/.default").token
# ヘッダは "名前: 値" 形式（コロン区切り）。--transport http も明示する。
# "Authorization=Bearer ..." のようにイコールで書くとヘッダとして解釈されず、

# CLI が OAuth ディスカバリにフォールバックして MCPOAuthError になります。print("copilot -i \"I just joined Zava as a senior engineer. According to company policy, how many vacation weeks does a senior employee get?\"")
print(f"copilot mcp add --transport http --header \"Authorization: Bearer {token}\" zava-kb \"{mcp_url}\"")

In [ ]:
mcp_url = f"{AZURE_SEARCH_SERVICE_ENDPOINT}/knowledgebases/{FILE_KNOWLEDGE_BASE_NAME}/mcp?api-version={AZURE_SEARCH_API_VERSION}"
token = credential.get_token("https://search.azure.com/.default").token
# ヘッダは "名前: 値" 形式（コロン区切り）。--transport http も明示する。
print(f"copilot mcp add --transport http --header \"Authorization: Bearer {token}\" sharepoint-itops-kb \"{mcp_url}\"")
print("copilot -i \"KINTAI-X の打刻反映遅延で確認すべきキューとジョブを教えて。\"")

copilot mcp add --transport http --header "Authorization: Bearer ***REMOVED-LEAKED-TOKEN***" sharepoint-itops-kb "https://gi-ai-search.search.windows.net/knowledgebases/sharepoint-it-ops-knowledge-base/mcp?api-version=2026-05-01-preview"
copilot -i "KINTAI-X の打刻反映遅延で確認すべきキューとジョブを教えて。"


: 

## ✅ ミッション達成

**構築したもの:**

* ✓ **File Knowledge Source**: PDF を直接アップロードでき、チャンク化と埋め込み生成を自動で行うソース。
* ✓ **Indexed Knowledge Sources**: ビルド済みの HR と health の検索インデックス用ソース。インデックススキーマと処理を完全に制御できる。
* ✓ **マルチソース Knowledge Base**: 各サブクエスチョンを自動で適切なソースへルーティングし、引用付きの回答を返す単一の KB。

➡️ [Part 2: Web 検索結果からの grounding を追加する](part2-search-mcp-kb.ipynb) に進みましょう。